In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

RAW_RATINGS_PATH = ROOT / "data" / "raw" / "movie-ratings.txt"
CLEANED_RATINGS_PATH = ROOT / "data" / "processed" / "cleaned_ratings.csv"


In [ ]:
column_names = [
    "userId",
    "movieId",
    "categoryId",
    "reviewId",
    "rating",
    "reviewDate"
]

df = pd.read_csv(
    RAW_RATINGS_PATH,
    sep=",",
    names=column_names,
    header=None
)

print("Original Shape:", df.shape)

df = df.drop_duplicates()

print("After Removing Duplicates:", df.shape)

print("\nMissing Values:")
print(df.isnull().sum())

CLEANED_RATINGS_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(CLEANED_RATINGS_PATH, index=False)

print("\nCleaned dataset saved successfully.")


##**Item-Based CF**

In [ ]:
def build_item_user_matrix(train):
    user_codes, users = pd.factorize(train["userId"], sort=True)
    item_codes, items = pd.factorize(train["movieId"], sort=True)
    ratings = train["rating"].astype(float).to_numpy()

    item_user = csr_matrix(
        (ratings, (item_codes, user_codes)),
        shape=(len(items), len(users))
    )
    item_vectors = normalize(item_user, norm="l2", axis=1)
    item_lookup = {movie_id: idx for idx, movie_id in enumerate(items)}

    user_items = (
        pd.DataFrame({
            "userId": train["userId"].to_numpy(),
            "item_idx": item_codes,
            "rating": ratings,
        })
        .groupby("userId")[["item_idx", "rating"]]
        .apply(lambda x: (x["item_idx"].to_numpy(), x["rating"].to_numpy()))
        .to_dict()
    )

    return item_vectors, item_lookup, user_items


def predict_item_cf(test, item_vectors, item_lookup, user_items, global_mean):
    predictions = []

    for row in test.itertuples(index=False):
        movie_idx = item_lookup.get(row.movieId)
        history = user_items.get(row.userId)

        if movie_idx is None or history is None:
            predictions.append(global_mean)
            continue

        rated_item_indices, user_ratings = history
        sims = item_vectors[movie_idx].dot(item_vectors[rated_item_indices].T).toarray().ravel()
        denom = np.abs(sims).sum()

        if denom == 0:
            predictions.append(global_mean)
        else:
            centered = user_ratings - global_mean
            predictions.append(global_mean + float(np.dot(sims, centered) / denom))

    return np.clip(np.array(predictions), 1, 5)

print("Item-Based CF helper functions loaded.")


In [ ]:
ratings_df = df[["userId", "movieId", "rating"]]
train, test = train_test_split(ratings_df, test_size=0.2, random_state=42)

print("Train Shape:", train.shape)
print("Test Shape:", test.shape)


In [ ]:
global_avg = train["rating"].mean()
item_vectors, item_lookup, user_items = build_item_user_matrix(train)
item_predictions = predict_item_cf(test, item_vectors, item_lookup, user_items, global_avg)

itemcf_rmse = np.sqrt(mean_squared_error(test["rating"], item_predictions))
itemcf_mae = mean_absolute_error(test["rating"], item_predictions)

print("Item-Based Collaborative Filtering Performance")
print("RMSE:", round(itemcf_rmse, 4))
print("MAE:", round(itemcf_mae, 4))


##**Global Average**

In [ ]:
# The same train/test split is used for the Global Average baseline.


In [ ]:
print("Train Shape:", train.shape)
print("Test Shape:", test.shape)


In [ ]:
global_avg = train["rating"].mean()

print("\nGlobal Average Rating:", round(global_avg, 3))

global_predictions = np.full(len(test), global_avg)

global_rmse = np.sqrt(mean_squared_error(test["rating"], global_predictions))
global_mae = mean_absolute_error(test["rating"], global_predictions)

print("\nBaseline Model Performance")
print("RMSE:", round(global_rmse, 4))
print("MAE:", round(global_mae, 4))


##**Comparison**

In [ ]:
model_names = ["Global Average", "Item-Based CF"]
rmse_scores = [round(global_rmse, 4), round(itemcf_rmse, 4)]
mae_scores = [round(global_mae, 4), round(itemcf_mae, 4)]

performance_df = pd.DataFrame({
    "Model": model_names,
    "RMSE": rmse_scores,
    "MAE": mae_scores
})

print("\nModel Performance Comparison:")
display(performance_df)


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

rmse_colors = ["#3f6f8f", "#49a878"]
mae_colors = ["#8e2a92", "#df7f61"]

axes[0].bar(performance_df["Model"], performance_df["RMSE"], color=rmse_colors)
axes[0].set_title("RMSE Comparison")
axes[0].set_ylabel("RMSE Score")
axes[0].set_xlabel("")
axes[0].set_ylim(min(rmse_scores) * 0.9, max(rmse_scores) * 1.1)
for index, row in performance_df.iterrows():
    axes[0].text(index, row.RMSE, round(row.RMSE, 4), color="black", ha="center", va="bottom")

axes[1].bar(performance_df["Model"], performance_df["MAE"], color=mae_colors)
axes[1].set_title("MAE Comparison")
axes[1].set_ylabel("MAE Score")
axes[1].set_xlabel("")
axes[1].set_ylim(min(mae_scores) * 0.9, max(mae_scores) * 1.1)
for index, row in performance_df.iterrows():
    axes[1].text(index, row.MAE, round(row.MAE, 4), color="black", ha="center", va="bottom")

plt.tight_layout()
plt.show()


##**Conclusion**
Both algorithms seemed to have very similar performances, with the global average having a higher MAE but a lower RMSE compared to item-based CF. This could be due to item-based CF not being complex enough to have a much higher score than global average. The sparsity of reviews making it harder for item-based CF to find similar items as well as the distribution of the ratings being greatly skewed to high ratings resulting in an imbalanced dataset.